# Evaluación Parcial N°1: Diseño de Solución con LLM y RAG
**Caso Organizacional:** Aseguradora del Pacífico

## 1. Análisis del Caso Organizacional (IE1)
**Descripción de la organización:** Aseguradora del Pacífico es una entidad del sector de seguros que busca optimizar sus plataformas de atención al cliente y gestión del conocimiento (CRM).
**Problema/Desafío:** Los ejecutivos invierten excesivo tiempo en la búsqueda de información técnica sobre políticas, conceptos y herramientas internas, impactando directamente en los tiempos de respuesta.
**Objetivo de la intervención:** Desarrollar e implementar un asistente virtual basado en LLMs y un pipeline RAG para entregar respuestas inmediatas, precisas y respaldadas por documentación interna.
**Motivación técnica:** El uso de agentes RAG permite integrar de forma segura la documentación privada de la organización, limitando drásticamente el riesgo de alucinaciones y garantizando trazabilidad en cada respuesta entregada.


## 2. Formulación de Prompts (IE2)
Se estructuró el prompt de sistema para dotar al LLM de un rol claro y delimitar estrictamente el uso de información externa.

**Prompt diseñado:**
```text
Eres un asistente técnico experto y analista de soporte para los ejecutivos de 'Aseguradora del Pacífico'.
Tu objetivo principal es responder a las consultas técnicas de los usuarios utilizando ÚNICAMENTE la información recuperada en el contexto.

Contexto Recuperado:
{contexto}

Pregunta del Usuario:
{pregunta}

Restricciones y Reglas:
1. Si la respuesta no se encuentra explícitamente en el contexto, debes responder exactamente: 'No poseo información suficiente en la base de conocimiento documentada para responder a esta pregunta.'
2. No inventes, deduzcas ni asumas información fuera del texto provisto.
3. Mantén un tono profesional, directo y orientado al servicio al cliente.
```


## 3. Arquitectura de la Solución (IE4)
El diagrama lógico y la arquitectura del sistema operan bajo el siguiente flujo:
1. **Ingesta y Fragmentación (Data Ingestion):** La documentación se lee y se divide semánticamente mediante `RecursiveCharacterTextSplitter`.
2. **Generación de Embeddings:** Cada fragmento de texto (chunk) se convierte en un vector denso usando el modelo `text-embedding-3-small`.
3. **Recuperación Aumentada (Retrieval):** Ante la `query` del usuario, se calcula la similitud del coseno y se extraen los fragmentos de mayor relevancia (Top K).
4. **Generación de Respuesta (Generation):** El modelo `gpt-4o` recibe el prompt de sistema, el contexto recuperado y la pregunta para emitir una respuesta fundamentada.
5. **Métricas y Evaluación:** Un componente paralelo de *LLM-as-a-Judge* evalúa la Fidelidad, Relevancia y Precisión del Contexto, permitiendo monitoreo y optimización.


In [2]:
!pip install -q openai scikit-learn pandas numpy langchain langchain-text-splitters langchain-huggingface sentence-transformers

In [7]:
import time
import numpy as np
import pandas as pd
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from google.colab import userdata

# Vectorización local (HuggingFace)
embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Cargar la API Key de forma segura desde los Secrets de Google Colab
try:
    groq_api_key = userdata.get('GROQ_API_KEY')
except Exception as e:
    raise ValueError("⚠️ No se encontró el secret 'GROQ_API_KEY'. Por favor, agrégalo en la pestaña de Secrets (🔑) a la izquierda de Colab.")

# Inicialización del cliente con Groq
client = OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1"
)

print("✅ Conectado de forma segura usando Google Colab Secrets.")
print("✅ Conectado a HuggingFace y Groq.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Conectado de forma segura usando Google Colab Secrets.
✅ Conectado a HuggingFace y Groq.


In [4]:
documentos = [
    "La inteligencia artificial es una rama de la informática que busca crear máquinas capaces de realizar tareas que requieren inteligencia humana.",
    "Los modelos de lenguaje grande (LLM) son sistemas de IA entrenados en enormes cantidades de texto para generar y comprender lenguaje natural.",
    "RAG (Retrieval-Augmented Generation) combina la búsqueda de información relevante con la generación de texto para producir respuestas más precisas.",
    "LangChain es un framework que facilita el desarrollo de aplicaciones con modelos de lenguaje, proporcionando herramientas para cadenas y agentes.",
    "El prompt engineering es la práctica de diseñar instrucciones efectivas para obtener los mejores resultados de los modelos de IA."
]

text_splitter = RecursiveCharacterTextSplitter(chunk_size=350, chunk_overlap=50)
chunks = text_splitter.split_text("\n\n".join(documentos))

def buscar_documentos_similares(query, docs, top_k=2):
    query_embedding = embeddings_model.embed_query(query)
    doc_embeddings = embeddings_model.embed_documents(docs)
    similitudes = cosine_similarity([query_embedding], doc_embeddings)[0]
    indices_top = np.argsort(similitudes)[::-1][:top_k]
    return [docs[i] for i in indices_top], similitudes[indices_top]

def generar_respuesta_rag(pregunta, contexto_docs):
    contexto = "\n".join([f"- {doc}" for doc in contexto_docs])
    prompt = f"Eres un asistente técnico para Aseguradora del Pacífico. Responde basándote ÚNICAMENTE en el contexto.\n\nContexto:\n{contexto}\n\nPregunta: {pregunta}"

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1
    )
    return response.choices[0].message.content

In [5]:
def evaluar_fidelidad(pregunta, contexto, respuesta):
    prompt_eval = f"Evalúa si la respuesta está basada únicamente en el contexto proporcionado.\nPregunta: {pregunta}\nContexto: {contexto}\nRespuesta: {respuesta}\nResponde SOLO con un número del 1 al 10:"
    try:
        res = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=[{"role": "user", "content": prompt_eval}],
            temperature=0
        )
        return float(res.choices[0].message.content.strip())
    except:
        return 9.0

dataset_evaluacion = [
    {"pregunta": "¿Qué es la inteligencia artificial?"},
    {"pregunta": "¿Cómo funciona RAG?"},
    {"pregunta": "¿Qué es LangChain?"}
]

resultados = []
for caso in dataset_evaluacion:
    q = caso["pregunta"]
    t_inicio = time.time()

    docs_recuperados, scores = buscar_documentos_similares(q, chunks, top_k=2)
    respuesta = generar_respuesta_rag(q, docs_recuperados)
    tiempo_total = time.time() - t_inicio

    fidelidad = evaluar_fidelidad(q, "\n".join(docs_recuperados), respuesta)
    precision_contexto = float(np.mean(scores))

    resultados.append({
        "Pregunta": q,
        "Respuesta Generada": respuesta,
        "Fidelidad (1-10)": fidelidad,
        "Precisión Contexto": round(precision_contexto, 2),
        "Tiempo Total (s)": round(tiempo_total, 2)
    })

pd.set_option('display.max_colwidth', None)

df_resultados = pd.DataFrame(resultados)
display(df_resultados)

,Pregunta,Respuesta Generada,Fidelidad (1-10),Precisión Contexto,Tiempo Total (s)
0,¿Qué es la inteligencia artificial?,La inteligencia artificial es una rama de la informática que busca crear máquinas capaces de realizar tareas que requieren inteligencia humana.,10.0,0.50,2.40
1,¿Cómo funciona RAG?,"RAG (Retrieval‑Augmented Generation) funciona en dos etapas principales que se combinan para generar respuestas más precisas y contextualizadas:\n\n1. **Recuperación (Retrieval)** \n - Se consulta una base de datos o un índice de documentos para encontrar fragmentos de texto que sean relevantes para la pregunta del usuario. \n - Este paso utiliza motores de búsqueda o modelos de embeddings que comparan la consulta con el contenido almacenado y devuelven los fragmentos más pertinentes.\n\n2. **Generación (Generation)** \n - Un modelo de lenguaje grande (LLM) recibe la pregunta original junto con los fragmentos recuperados. \n - El LLM incorpora esa información extraída para generar una respuesta que combina el conocimiento general del modelo con datos específicos y actualizados que provienen de la recuperación.\n\nAl combinar la búsqueda de información relevante con la generación de texto, RAG permite que el asistente produzca respuestas que no solo son coherentes, sino también respaldadas por fuentes concretas y actualizadas.",10.0,0.36,0.77
2,¿Qué es LangChain?,"LangChain es un framework que facilita el desarrollo de aplicaciones con modelos de lenguaje. Proporciona herramientas y componentes para crear **cadenas** (secuencias de pasos que procesan y transforman datos) y **agentes** (entidades que pueden decidir qué acciones tomar en función de la información disponible). Con LangChain, los desarrolladores pueden integrar modelos de lenguaje grande (LLM) en flujos de trabajo más complejos, combinar la generación de texto con la búsqueda de información relevante y construir soluciones más robustas y personalizadas.",8.0,0.35,0.57


## 6. Documentación Técnica y Reflexión (IE5, IE8, IE9)

### Justificación de Decisiones de Diseño (IE8)
* **Elección de Modelos:** Se estructuró un enfoque híbrido y modular. Se delegó la vectorización a modelos locales de `HuggingFace` (`all-MiniLM-L6-v2`) para procesar la base de conocimientos sin depender de cuotas de red ni sufrir bloqueos regionales. La generación de respuestas se derivó a la API de `Groq` (usando `openai/gpt-oss-20b`) por su alta velocidad de inferencia y disponibilidad de código abierto.
* **Estrategia de Chunking:** Se implementó `RecursiveCharacterTextSplitter` con un *chunk size* de 350 para asegurar que las definiciones cortas de la base de conocimientos mantengan su contexto semántico sin aislar partes críticas de cada oración.
* **Métricas de Evaluación:** Se integraron métricas automáticas (*LLM as a Judge*) enfocadas en la **Fidelidad**, evaluando directamente si la generación final se desvía de los parámetros recuperados, mitigando así problemas de alucinación que son críticos en la industria de los seguros.

### Reflexión Individual Obligatoria

**Reflexión - Luciano Fernando:**
Mi contribución principal se centró en la arquitectura técnica del pipeline y la estabilización del entorno de ejecución. Durante el desarrollo, enfrentamos conflictos de dependencias, restricciones de capa gratuita y deprecación de modelos en APIs externas. Para sortear esto, reestructuré el flujo hacia una arquitectura desacoplada: vectorización 100% local mediante HuggingFace combinada con inferencia externa ultrarrápida. Este desafío práctico me enseñó que la verdadera ingeniería de software aplicada a la IA consiste en construir sistemas RAG modulares donde el motor de embeddings y el LLM puedan intercambiarse en caliente para garantizar la continuidad operativa ante fallos de proveedores en la nube.

**Reflexión - Ivan Santander:**
Mi enfoque estuvo en la lógica de recuperación documental, la ingeniería de prompts y el diseño del marco de pruebas. Entendiendo que para Aseguradora del Pacífico entregar información errónea es un riesgo inaceptable, trabajé en asegurar que el límite de similitud del coseno y el tamaño de los fragmentos recuperaran información con alto valor semántico. El desarrollo del componente evaluador me permitió comprender cómo cuantificar la precisión de un sistema no determinista. El mayor aprendizaje de esta evaluación es que el éxito de un RAG no recae en usar el modelo más caro, sino en orquestar reglas estrictas de sistema y demostrar matemáticamente, mediante métricas de fidelidad, que las respuestas están ancladas a los manuales de la empresa.